#House Price Prediction Model

##Data

###Import

In [ ]:
import pandas as pd

df = pd.read_csv("/content/sample_data/Housing.csv")

: 

In [ ]:
df.head()

###Normalisation

In [ ]:
df = df.drop('id', axis=1)

In [ ]:
df['date'] = pd.to_datetime(df['date'])

In [ ]:
df['date'] = df['date'].dt.year

In [ ]:
df[['bedrooms', 'bathrooms']] = round(df[['bedrooms', 'bathrooms']], 0)

In [ ]:
df['age'] = df['date'] - df['yr_built']
#df['bath_per_sqft'] = df['bathrooms'] / df['sqft_living']
df['bed_per_bath'] = df['bedrooms'] / (df['bathrooms'] + 0.01)
df['time_since_renovated'] = df['date'] - df['yr_renovated']
df['rooms'] = df['bathrooms'] + df['bedrooms']
df['sqft_per_room'] = df['sqft_living'] / (df['rooms'] + 0.1)

In [ ]:
df = df.iloc[:, [0,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,1]]

In [ ]:
df.head()

In [ ]:
df.info()

###Dataset split

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop('price', axis=1)
Y = df['price']

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)

In [ ]:
x_train.info()

##Gradient Descent

In [ ]:
import xgboost as xgb

model = xgb.XGBRegressor(n_estimators = 1500, learning_rate = 0.06, max_depth = 5)

#Training
model.fit (x_train, y_train, eval_set=[(x_test, y_test)], verbose=100)

#Prediction
y_pred = model.predict(x_test)

from sklearn.metrics import mean_squared_error

mse = mean_squared_error(y_test, y_pred)

print(f"Mean Squared Error: {mse}")

##Results


In [ ]:
import numpy as np

results = pd.DataFrame({
    'Actual Price': y_test,
    'Predicted Price': y_pred,
    'Absolute Error': np.abs(y_test - y_pred),
    'Percentage Error': np.abs((y_test - y_pred) / y_test) * 100
})

formatted_results = results.copy()

formatted_results['Actual Price'] = formatted_results['Actual Price'].map('${:,.2f}'.format)
formatted_results['Predicted Price'] = formatted_results['Predicted Price'].map('${:,.2f}'.format)
formatted_results['Absolute Error'] = formatted_results['Absolute Error'].map('${:,.2f}'.format)
formatted_results['Percentage Error'] = formatted_results['Percentage Error'].map('{:,.2f}%'.format)

In [ ]:
formatted_results.head(20)

In [ ]:
total_percentage_error = np.mean(results['Percentage Error'])
median_ape = np.median(results['Percentage Error'])

print(f"Median Percentage Error: {median_ape:.2f}%")
print(f'Mean Percentage Error: {total_percentage_error:.2f}%')